[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/RL_tmp/blob/main/notebooks/double_integrator_dqn_colab.ipynb)

# Double Integrator — DQN control

목표는 1차원 질점을 원점에 정지시키는 것이다.

상태공간은
\[
X=\mathbb{R}^2,\qquad x=(q,v)\in X
\]
이고, \(q\)는 위치, \(v\)는 속도다.

제어 입력은 세 개의 힘 중 하나로 제한한다.
\[
U=\{-1,0,+1\},\qquad \pi_\theta:X\to U
\]

동역학은
\[
\dot q=v,\qquad \dot v=u
\]
이며 시간 간격 \(\Delta t\)로 계산할 때
\[
q_{t+1}=q_t+v_t\Delta t,\qquad
v_{t+1}=v_t+u_t\Delta t
\]
를 사용한다.

학습 중 일정한 episode 간격마다 현재 정책으로 평가 episode를 한 번 실행한다. 왼쪽은 실제 질점의 움직임, 오른쪽은 같은 상태 \(x_t=(q_t,v_t)\)가 상태공간에서 움직이는 궤적이다.

In [ ]:
import math
import random
from collections import deque
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display, Image

# Reproducibility
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Environment
DT = 0.08
MAX_STEPS = 100
FORCES = np.array([-1.0, 0.0, 1.0], dtype=np.float32)

# Training
EPISODES = 300
BATCH_SIZE = 64
GAMMA = 0.98
LR = 1e-3
REPLAY_SIZE = 20_000
WARMUP = 500
TARGET_UPDATE = 250
EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY_STEPS = 12_000

# Evaluation / rendering
EVAL_EVERY = 25          # 1이면 매 training episode 뒤에 평가
GIF_FPS = 20             # 저장할 GIF의 초당 프레임 수
OUTPUT_DIR = Path("double_integrator_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
class DoubleIntegratorEnv:
    def __init__(self, dt=DT, max_steps=MAX_STEPS):
        self.dt = dt
        self.max_steps = max_steps
        self.state = None
        self.steps = 0

    def reset(self, q=None, v=None):
        if q is None:
            q = np.random.uniform(-2.0, 2.0)
        if v is None:
            v = np.random.uniform(-0.5, 0.5)
        self.state = np.array([q, v], dtype=np.float32)
        self.steps = 0
        return self.state.copy()

    def step(self, action_index):
        q, v = self.state
        u = float(FORCES[action_index])

        q_next = q + v * self.dt
        v_next = v + u * self.dt

        self.state = np.array([q_next, v_next], dtype=np.float32)
        self.steps += 1

        # 목표 (q,v)=(0,0)으로 가도록 하는 비용
        cost = q_next**2 + 0.25 * v_next**2 + 0.01 * u**2
        reward = -float(cost)

        success = abs(q_next) < 0.05 and abs(v_next) < 0.05
        done = success or self.steps >= self.max_steps

        return self.state.copy(), reward, done, {"success": success, "u": u}


env = DoubleIntegratorEnv()
env.reset()


In [ ]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, len(FORCES)),
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

q_net = QNetwork().to(device)
target_net = QNetwork().to(device)
target_net.load_state_dict(q_net.state_dict())
target_net.eval()

optimizer = optim.Adam(q_net.parameters(), lr=LR)
replay = deque(maxlen=REPLAY_SIZE)


def epsilon_by_step(step):
    t = min(step / EPS_DECAY_STEPS, 1.0)
    return EPS_START + t * (EPS_END - EPS_START)


def select_action(state, epsilon):
    if random.random() < epsilon:
        return random.randrange(len(FORCES))
    x = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        return int(q_net(x).argmax(dim=1).item())


def optimize_once():
    if len(replay) < max(WARMUP, BATCH_SIZE):
        return None

    batch = random.sample(replay, BATCH_SIZE)
    states, actions, rewards, next_states, dones = map(np.array, zip(*batch))

    states = torch.tensor(states, dtype=torch.float32, device=device)
    actions = torch.tensor(actions, dtype=torch.int64, device=device).unsqueeze(1)
    rewards = torch.tensor(rewards, dtype=torch.float32, device=device)
    next_states = torch.tensor(next_states, dtype=torch.float32, device=device)
    dones = torch.tensor(dones, dtype=torch.float32, device=device)

    q = q_net(states).gather(1, actions).squeeze(1)

    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1).values
        target = rewards + GAMMA * (1.0 - dones) * next_q

    loss = nn.functional.smooth_l1_loss(q, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return float(loss.item())


In [ ]:
def run_evaluation_episode(q0=1.6, v0=0.0):
    eval_env = DoubleIntegratorEnv()
    state = eval_env.reset(q=q0, v=v0)

    trajectory = [state.copy()]
    forces = []

    for _ in range(MAX_STEPS):
        action = select_action(state, epsilon=0.0)
        next_state, reward, done, info = eval_env.step(action)
        forces.append(info["u"])
        trajectory.append(next_state.copy())
        state = next_state
        if done:
            break

    return np.asarray(trajectory), np.asarray(forces)


def render_episode_gif(trajectory, forces, training_episode, save_path=None, fps=GIF_FPS):
    if save_path is None:
        tag = str(training_episode).replace(" ", "_")
        save_path = OUTPUT_DIR / f"double_integrator_eval_{tag}.gif"
    else:
        save_path = Path(save_path)

    tr = np.asarray(trajectory)
    forces = np.asarray(forces)
    n_frames = max(1, len(forces))

    fig, (ax_phys, ax_state) = plt.subplots(1, 2, figsize=(11, 4.2))

    def update(frame_idx):
        idx_state = min(frame_idx + 1, len(tr) - 1)
        q, v = tr[idx_state]
        u = float(forces[min(frame_idx, len(forces) - 1)]) if len(forces) > 0 else 0.0

        ax_phys.clear()
        ax_phys.axvline(0.0, linestyle="--", linewidth=1)
        ax_phys.plot([-2.5, 2.5], [0, 0], linewidth=1)
        ax_phys.scatter([q], [0], s=180)
        ax_phys.scatter([0], [0], marker="x", s=120)
        ax_phys.arrow(
            q, 0.16,
            0.28 * u, 0,
            head_width=0.06,
            head_length=0.08,
            length_includes_head=True,
        )
        ax_phys.set_xlim(-2.5, 2.5)
        ax_phys.set_ylim(-0.45, 0.45)
        ax_phys.set_yticks([])
        ax_phys.set_xlabel("position q")
        ax_phys.set_title(
            f"Physical motion\nq={q:+.2f}, v={v:+.2f}, u={u:+.0f}"
        )

        ax_state.clear()
        ax_state.axhline(0.0, linewidth=0.8)
        ax_state.axvline(0.0, linewidth=0.8)
        ax_state.plot(tr[: idx_state + 1, 0], tr[: idx_state + 1, 1], marker="o", markersize=2)
        ax_state.scatter([tr[idx_state, 0]], [tr[idx_state, 1]], s=70)
        ax_state.scatter([0], [0], marker="x", s=120)
        ax_state.set_xlim(-2.5, 2.5)
        ax_state.set_ylim(-2.0, 2.0)
        ax_state.set_xlabel("position q")
        ax_state.set_ylabel("velocity v")
        ax_state.set_title("State-space trajectory (q, v)")
        ax_state.grid(alpha=0.25)

        fig.suptitle(
            f"Training episode {training_episode} | evaluation step {frame_idx + 1}",
            fontsize=12,
        )
        fig.tight_layout()
        return []

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=n_frames,
        interval=1000 / max(1, fps),
        blit=False,
        repeat=False,
    )
    anim.save(save_path, writer=animation.PillowWriter(fps=fps))
    plt.close(fig)
    return str(save_path)


def evaluate_and_save_gif(training_episode, q0=1.6, v0=0.0, fps=GIF_FPS):
    trajectory, forces = run_evaluation_episode(q0=q0, v0=v0)
    gif_path = render_episode_gif(
        trajectory,
        forces,
        training_episode=training_episode,
        fps=fps,
    )
    print(f"saved evaluation GIF: {gif_path}")
    display(Image(filename=gif_path))
    return trajectory, forces, gif_path


In [ ]:
episode_returns = []
loss_history = []
global_step = 0

for episode in range(1, EPISODES + 1):
    state = env.reset()
    total_reward = 0.0

    for _ in range(MAX_STEPS):
        epsilon = epsilon_by_step(global_step)
        action = select_action(state, epsilon)

        next_state, reward, done, info = env.step(action)
        replay.append((state, action, reward, next_state, done))

        loss = optimize_once()
        if loss is not None:
            loss_history.append(loss)

        global_step += 1
        if global_step % TARGET_UPDATE == 0:
            target_net.load_state_dict(q_net.state_dict())

        state = next_state
        total_reward += reward

        if done:
            break

    episode_returns.append(total_reward)

    if episode == 1 or episode % EVAL_EVERY == 0:
        trajectory, forces, gif_path = evaluate_and_save_gif(episode)

        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(episode_returns)
        ax.set_xlabel("training episode")
        ax.set_ylabel("return")
        ax.set_title(
            f"DQN training | episode={episode} | epsilon={epsilon_by_step(global_step):.3f}"
        )
        ax.grid(alpha=0.25)
        display(fig)
        plt.close(fig)

print("training finished")


In [ ]:
trajectory, forces, gif_path = evaluate_and_save_gif(
    training_episode="final",
    q0=1.6,
    v0=0.0,
)
print(gif_path)
